<a href="https://colab.research.google.com/github/thitivudb-lang/Motor-Health-Monitoring-System/blob/main/motor_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# CELL 1 — ติดตั้ง Library ทั้งหมด
# ==========================================
# firebase-admin  → เชื่อมต่อ Firebase
# scikit-learn    → เทรนโมเดล AI
# pandas          → จัดการข้อมูลตาราง
# numpy           → คำนวณตัวเลข
# matplotlib      → วาดกราฟ
# streamlit       → สร้าง Dashboard
# pyngrok         → เปิด URL สาธารณะ
# joblib          → บันทึกโมเดล

!pip install firebase-admin scikit-learn pandas numpy matplotlib streamlit pyngrok joblib -q
print("✅ ติดตั้งเสร็จทั้งหมด!")

✅ ติดตั้งเสร็จทั้งหมด!


In [ ]:
# ==========================================
# CELL 2 — อัพโหลด Service Account JSON
# ==========================================
# Service Account คือไฟล์กุญแจสำหรับเข้าถึง Firebase
# ต้องดาวน์โหลดจาก Firebase Console ก่อนนะคะ
# Project Settings → Service accounts → Generate new private key

from google.colab import files

print("กรุณาเลือกไฟล์ Service Account JSON ค่ะ...")
uploaded = files.upload()

# เก็บชื่อไฟล์ไว้ใช้ต่อ
SERVICE_ACCOUNT_FILE = list(uploaded.keys())[0]
print(f"✅ อัพโหลดไฟล์: {SERVICE_ACCOUNT_FILE} เสร็จแล้วค่ะ!")

กรุณาเลือกไฟล์ Service Account JSON ค่ะ...


Saving motor2-a75b1-firebase-adminsdk-fbsvc-5ecd04f922.json to motor2-a75b1-firebase-adminsdk-fbsvc-5ecd04f922 (1).json
✅ อัพโหลดไฟล์: motor2-a75b1-firebase-adminsdk-fbsvc-5ecd04f922 (1).json เสร็จแล้วค่ะ!


In [ ]:
# CELL 3 — เชื่อมต่อ Firebase
import firebase_admin
from firebase_admin import credentials, db

DATABASE_URL = "https://motor2-a75b1-default-rtdb.firebaseio.com/"

if not firebase_admin._apps:
    cred = credentials.Certificate(SERVICE_ACCOUNT_FILE)
    firebase_admin.initialize_app(cred, {"databaseURL": DATABASE_URL})
    print("✅ เชื่อมต่อ Firebase สำเร็จ!")
else:
    print("✅ Firebase เชื่อมต่ออยู่แล้วค่ะ!")

✅ Firebase เชื่อมต่ออยู่แล้วค่ะ!


In [ ]:
# ==========================================
# CELL 3.5 — ตรวจสอบ path ใน Firebase
# ==========================================
# ดึงข้อมูลทั้งหมดจาก root ก่อน
# เพื่อดูว่า path จริงๆ ชื่ออะไร

root_ref = db.reference("/")
all_data = root_ref.get()
print("ข้อมูลทั้งหมดใน Firebase:")
print(all_data)

In [ ]:
# CELL 4 — ดึงข้อมูล + สร้าง DataFrame
import pandas as pd
import numpy as np

ref = db.reference("motor")
snap = ref.get()

SENSOR_KEYS = ["ax", "ay", "az", "current", "rpm", "temperature", "vibration", "total", "timestamp"]

def extract_row(data):
    row = {}
    for key in SENSOR_KEYS:
        val = data.get(key, 0)
        if isinstance(val, dict):
            row[key] = 0
        else:
            try:
                row[key] = float(val)
            except (TypeError, ValueError):
                row[key] = 0
    return row

rows = []
if isinstance(snap, dict):
    for key, val in snap.items():
        if key in ["command", "history"]:
            continue
        if isinstance(val, dict):
            rows.append(extract_row(val))
    if len(rows) == 0:
        rows.append(extract_row(snap))

df = pd.DataFrame(rows)
print(f"✅ สร้าง DataFrame สำเร็จ: {df.shape}")
print(df.head())

✅ สร้าง DataFrame สำเร็จ: (1, 9)
    ax   ay   az  current    rpm  temperature  vibration    total  \
0  0.0  0.0  0.0 -0.07872  180.0      30.8125   10.30466  20582.0   

      timestamp  
0  1.782742e+09  


In [ ]:
# ============================================
# CELL 5 — Preprocessing
# ============================================
from sklearn.preprocessing import StandardScaler

FEATURES = ["ax", "ay", "az", "current", "rpm", "vibration"]

# ตรวจว่า df มีข้อมูลพอ
if df.empty or len(df) < 10:
    print("⚠️ ข้อมูลน้อยเกินไป — จะใช้ข้อมูลจาก history แทน")

    # ดึง history มาเสริม
    history = snap.get("history", {})
    hist_rows = []
    if isinstance(history, dict):
        for k, v in history.items():
            if isinstance(v, dict):
                hist_rows.append(extract_row(v))

    df = pd.DataFrame(hist_rows) if hist_rows else df
    print(f"ใช้ข้อมูลจาก history: {len(df)} rows")

X = df[FEATURES].dropna()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("✅ Preprocessing เสร็จแล้วค่ะ!")
print(f"Features ที่ใช้: {FEATURES}")
print(f"จำนวนข้อมูล: {X_scaled.shape[0]} แถว")
print(f"จำนวน Features: {X_scaled.shape[1]} ค่า")

⚠️ ข้อมูลน้อยเกินไป — จะใช้ข้อมูลจาก history แทน
ใช้ข้อมูลจาก history: 808 rows
✅ Preprocessing เสร็จแล้วค่ะ!
Features ที่ใช้: ['ax', 'ay', 'az', 'current', 'rpm', 'vibration']
จำนวนข้อมูล: 808 แถว
จำนวน Features: 6 ค่า


In [ ]:
# CELL 6 — เทรน Isolation Forest
from sklearn.ensemble import IsolationForest

model = IsolationForest(
    contamination=0.05,
    n_estimators=200,
    random_state=42
)

model.fit(X_scaled)
print("✅ เทรนโมเดลเสร็จแล้วค่ะ!")

predictions = model.predict(X_scaled)
health_scores = model.decision_function(X_scaled)

health_normalized = ((health_scores - health_scores.min()) /
                     (health_scores.max() - health_scores.min()) * 100).round(1)

n_normal = sum(predictions == 1)
n_fault  = sum(predictions == -1)

print(f"\nผลการตรวจสอบกับข้อมูล {len(predictions)} แถว:")
print(f"✅ ปกติ:   {n_normal} แถว ({n_normal/len(predictions)*100:.1f}%)")
print(f"❌ ผิดปกติ: {n_fault} แถว ({n_fault/len(predictions)*100:.1f}%)")
print(f"\nคะแนนสุขภาพเฉลี่ย: {health_normalized.mean():.1f}/100")
print(f"คะแนนสุขภาพต่ำสุด: {health_normalized.min():.1f}/100")
print(f"คะแนนสุขภาพสูงสุด: {health_normalized.max():.1f}/100")

✅ เทรนโมเดลเสร็จแล้วค่ะ!

ผลการตรวจสอบกับข้อมูล 808 แถว:
✅ ปกติ:   767 แถว (94.9%)
❌ ผิดปกติ: 41 แถว (5.1%)

คะแนนสุขภาพเฉลี่ย: 85.8/100
คะแนนสุขภาพต่ำสุด: 0.0/100
คะแนนสุขภาพสูงสุด: 100.0/100


In [ ]:
# CELL 7 — เก็บข้อมูลจริง 5000 rows
import time
import pandas as pd

ref = db.reference("motor")

print("🔄 เริ่มเก็บข้อมูล...")
print("⚠️ ตรวจสอบให้แน่ใจว่ามอเตอร์กำลังหมุนอยู่นะคะ!")
print("-" * 50)

rows = []
TARGET = 5000

for i in range(TARGET):
    snap = ref.get()
    if snap:
        data = snap.get("current", {})
        if isinstance(data, dict):
            row = {
                "current":     float(data.get("current", 0)),
                "rpm":         float(data.get("rpm", 0)),
                "temperature": float(data.get("temperature", 0)),
                "vibration":   float(data.get("vibration", 0)),
                "total":       float(data.get("total", 0)),
                "timestamp":   float(data.get("timestamp", 0)),
            }
            rows.append(row)

    if (i+1) % 100 == 0:
        print(f"เก็บแล้ว {i+1}/{TARGET} | rpm: {row.get('rpm',0)} | vibration: {row.get('vibration',0):.2f}")

    time.sleep(0.3)

df_new = pd.DataFrame(rows)
print("-" * 50)
print(f"✅ เก็บข้อมูลได้ {len(df_new)} แถว!")
print(df_new[["current","rpm","temperature","vibration"]].describe().round(2))

🔄 เริ่มเก็บข้อมูล...
⚠️ ตรวจสอบให้แน่ใจว่ามอเตอร์กำลังหมุนอยู่นะคะ!
--------------------------------------------------
เก็บแล้ว 100/5000 | rpm: 120.0 | vibration: 10.36
เก็บแล้ว 200/5000 | rpm: 2280.0 | vibration: 10.17
เก็บแล้ว 300/5000 | rpm: 2280.0 | vibration: 10.17
เก็บแล้ว 400/5000 | rpm: 180.0 | vibration: 10.62
เก็บแล้ว 500/5000 | rpm: 240.0 | vibration: 10.16
เก็บแล้ว 600/5000 | rpm: 120.0 | vibration: 10.36
เก็บแล้ว 700/5000 | rpm: 1140.0 | vibration: 10.28
เก็บแล้ว 800/5000 | rpm: 180.0 | vibration: 10.42
เก็บแล้ว 900/5000 | rpm: 240.0 | vibration: 10.35
เก็บแล้ว 1000/5000 | rpm: 240.0 | vibration: 10.35
เก็บแล้ว 1100/5000 | rpm: 120.0 | vibration: 10.27
เก็บแล้ว 1200/5000 | rpm: 240.0 | vibration: 10.13
เก็บแล้ว 1300/5000 | rpm: 360.0 | vibration: 10.23
เก็บแล้ว 1400/5000 | rpm: 300.0 | vibration: 10.56
เก็บแล้ว 1500/5000 | rpm: 180.0 | vibration: 10.16
เก็บแล้ว 1600/5000 | rpm: 2220.0 | vibration: 10.23
เก็บแล้ว 1700/5000 | rpm: 120.0 | vibration: 10.46
เก็บแล้ว 1800/5000 

In [ ]:
# เช็คสถิติข้อมูลปัจจุบัน
print("=== สถิติข้อมูลที่มี ===")
print(f"จำนวน rows: {len(df)}")
print()
print(df[["ax","ay","az","current","rpm","vibration","temperature"]].describe().round(2))

=== สถิติข้อมูลที่มี ===
จำนวน rows: 808

          ax     ay     az  current        rpm  vibration  temperature
count  808.0  808.0  808.0   808.00     808.00     808.00       808.00
mean     0.0    0.0    0.0    -0.71    1483.22      10.18        31.89
std      0.0    0.0    0.0     1.25   23454.44       0.29         1.85
min      0.0    0.0    0.0    -4.05       0.00       8.90        28.06
25%      0.0    0.0    0.0    -0.60       0.00      10.08        30.69
50%      0.0    0.0    0.0    -0.15       0.00      10.18        32.62
75%      0.0    0.0    0.0    -0.08       0.00      10.22        32.75
max      0.0    0.0    0.0     0.97  564000.00      12.51        42.31


In [ ]:
# ตรวจสอบโครงสร้างข้อมูลจริงๆ
ref = db.reference("motor")
snap = ref.get()

print("=== โครงสร้างทั้งหมด ===")
for key, val in snap.items():
    if key == "history":
        continue
    print(f"{key}: {val} (type: {type(val).__name__})")

=== โครงสร้างทั้งหมด ===
command: ON (type: str)
current: {'current': 0.03655, 'motor': 'ON', 'rpm': 120, 'temperature': 30.25, 'timestamp': 1782745047, 'total': 2147, 'vibration': 10.03841} (type: dict)


In [ ]:
# CELL 8 — เทรนโมเดลใหม่ด้วยข้อมูลจริง
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
import joblib

FEATURES = ["current", "rpm", "temperature", "vibration"]

# ใช้ df_new จาก Cell 7
X_new = df_new[FEATURES].dropna()
scaler_new = StandardScaler()
X_new_scaled = scaler_new.fit_transform(X_new)

model_new = IsolationForest(
    contamination=0.05,
    n_estimators=200,
    random_state=42
)
model_new.fit(X_new_scaled)

# ตรวจสอบผล
predictions_new = model_new.predict(X_new_scaled)
health_scores = model_new.decision_function(X_new_scaled)
health_normalized = ((health_scores - health_scores.min()) /
                     (health_scores.max() - health_scores.min()) * 100).round(1)

n_normal = sum(predictions_new == 1)
n_fault  = sum(predictions_new == -1)

print("✅ เทรนโมเดลใหม่เสร็จแล้วค่ะ!")
print(f"ปกติ:    {n_normal} แถว ({n_normal/len(predictions_new)*100:.1f}%)")
print(f"ผิดปกติ: {n_fault} แถว ({n_fault/len(predictions_new)*100:.1f}%)")
print(f"คะแนนสุขภาพเฉลี่ย: {health_normalized.mean():.1f}/100")

# บันทึกโมเดล
joblib.dump(model_new, "/content/motor_model_v2.pkl")
joblib.dump(scaler_new, "/content/scaler_v2.pkl")
print("\n✅ บันทึกโมเดลเสร็จ!")
print("Features ที่ใช้:", FEATURES)

✅ เทรนโมเดลใหม่เสร็จแล้วค่ะ!
ปกติ:    4751 แถว (95.0%)
ผิดปกติ: 249 แถว (5.0%)
คะแนนสุขภาพเฉลี่ย: 82.2/100

✅ บันทึกโมเดลเสร็จ!
Features ที่ใช้: ['current', 'rpm', 'temperature', 'vibration']


In [ ]:
from google.colab import files

files.download("/content/motor_model_v2.pkl")
files.download("/content/scaler_v2.pkl")
print("✅ ดาวน์โหลดเสร็จ! พร้อมส่งให้ทีม 4B ค่ะ")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ ดาวน์โหลดเสร็จ! พร้อมส่งให้ทีม 4B ค่ะ
